In [1]:
import os
import zipfile
import numpy as np
import tensorflow as tf

from tensorflow.keras.layers import (
    Input,
    Conv1D,
    MaxPooling1D,
    GlobalAveragePooling1D,
    Concatenate,
    Dense
)

from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


# ============================================================
# 1. Seed
# ============================================================

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# 2. Path
# ============================================================

DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/UCI HAR Dataset"


# ============================================================
# 3. Download
# ============================================================

if not os.path.exists(DATA_PATH):

    url = (
        "https://archive.ics.uci.edu/ml/machine-learning-databases/"
        "00240/UCI%20HAR%20Dataset.zip"
    )

    zip_path = tf.keras.utils.get_file(
        "UCI_HAR.zip",
        origin=url,
        cache_dir="/content",
        cache_subdir=""
    )

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(
            "/content/drive/MyDrive/Colab Notebooks"
        )


# ============================================================
# 4. Signals
# ============================================================

signals = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
    "total_acc_x",
    "total_acc_y",
    "total_acc_z"
]


# ============================================================
# 5. Load Dataset
# ============================================================

def load_data(split):

    X = []

    for signal in signals:

        file_path = (
            f"{DATA_PATH}/{split}/"
            f"Inertial Signals/{signal}_{split}.txt"
        )

        X.append(
            np.loadtxt(file_path)
        )

    X = np.stack(
        X,
        axis=-1
    )

    y = np.loadtxt(
        f"{DATA_PATH}/{split}/"
        f"y_{split}.txt"
    ).astype(int)

    y = y - 1

    return X, y


X_train, y_train = load_data("train")
X_test, y_test = load_data("test")


# ============================================================
# 6. Normalization
# ============================================================

mean = X_train.mean(
    axis=(0, 1),
    keepdims=True
)

std = X_train.std(
    axis=(0, 1),
    keepdims=True
)

X_train = (
    X_train - mean
) / (
    std + 1e-8
)

X_test = (
    X_test - mean
) / (
    std + 1e-8
)


# ============================================================
# 7. One-hot
# ============================================================

y_train_onehot = to_categorical(
    y_train,
    6
)

y_test_onehot = to_categorical(
    y_test,
    6
)


# ============================================================
# 8. Fire Module
# ============================================================

def fire_module(
    x,
    squeeze_filters,
    expand_filters
):

    # Squeeze

    squeeze = Conv1D(
        squeeze_filters,
        1,
        padding="same",
        activation="relu"
    )(x)


    # Expand 1x1

    expand_1x1 = Conv1D(
        expand_filters,
        1,
        padding="same",
        activation="relu"
    )(squeeze)


    # Expand 3x3

    expand_3x3 = Conv1D(
        expand_filters,
        3,
        padding="same",
        activation="relu"
    )(squeeze)


    # Concatenate

    x = Concatenate()([
        expand_1x1,
        expand_3x3
    ])

    return x


# ============================================================
# 9. SqueezeNet
# ============================================================

inputs = Input(
    shape=(128, 9)
)


x = Conv1D(
    32,
    3,
    padding="same",
    activation="relu"
)(inputs)


x = fire_module(
    x,
    16,
    32
)

x = MaxPooling1D(
    2
)(x)


x = fire_module(
    x,
    32,
    64
)

x = MaxPooling1D(
    2
)(x)


x = fire_module(
    x,
    64,
    128
)


x = GlobalAveragePooling1D()(x)


x = Dense(
    128,
    activation="relu"
)(x)


outputs = Dense(
    6,
    activation="softmax"
)(x)


model = Model(
    inputs,
    outputs
)


# ============================================================
# 10. Compile
# ============================================================

model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)


model.summary()


# ============================================================
# 11. Train
# ============================================================

history = model.fit(
    X_train,
    y_train_onehot,
    epochs=20,
    batch_size=64,
    validation_split=0.2,
    shuffle=True,
    verbose=1
)


# ============================================================
# 12. Prediction
# ============================================================

y_prob = model.predict(
    X_test,
    verbose=0
)

y_pred = np.argmax(
    y_prob,
    axis=1
)


# ============================================================
# 13. Evaluation
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)


print("\n==============================")
print("SQUEEZENET TEST RESULTS")
print("==============================")

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")


# ============================================================
# 14. Classification Report
# ============================================================

class_names = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING"
]


print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


# ============================================================
# 15. Confusion Matrix
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred
)

print("\nConfusion Matrix")

print(cm)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 9)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 128, 32)   │        896 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 128, 16)   │        528 │ conv1d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 128, 32)   │        544 │ conv1d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 128, 32)   │      1,568 │ conv1d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 128, 64)   │          0 │ conv1d_2[0][0],   │
│ (Concatenate)       │                   │            │ conv1d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 64, 64)    │          0 │ concatenate[0][0] │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 64, 32)    │      2,080 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 64, 64)    │      2,112 │ conv1d_4[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_6 (Conv1D)   │ (None, 64, 64)    │      6,208 │ conv1d_4[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 64, 128)   │          0 │ conv1d_5[0][0],   │
│ (Concatenate)       │                   │            │ conv1d_6[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 32, 128)   │          0 │ concatenate_1[0]… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_7 (Conv1D)   │ (None, 32, 64)    │      8,256 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_8 (Conv1D)   │ (None, 32, 128)   │      8,320 │ conv1d_7[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_9 (Conv1D)   │ (None, 32, 128)   │     24,704 │ conv1d_7[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 32, 256)   │          0 │ conv1d_8[0][0],   │
│ (Concatenate)       │                   │            │ conv1d_9[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 256)       │          0 │ concatenate_2[0]… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     32,896 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 6)         │        774 │ dense[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 88,886 (347.21 KB)

 Trainable params: 88,886 (347.21 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 16s 75ms/step - accuracy: 0.7181 - loss: 0.6751 - val_accuracy: 0.8987 - val_loss: 0.3398
Epoch 2/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9260 - loss: 0.1888 - val_accuracy: 0.9143 - val_loss: 0.2568
Epoch 3/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9449 - loss: 0.1275 - val_accuracy: 0.9096 - val_loss: 0.2483
Epoch 4/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9548 - loss: 0.1067 - val_accuracy: 0.9082 - val_loss: 0.2591
Epoch 5/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9565 - loss: 0.0926 - val_accuracy: 0.9069 - val_loss: 0.2841
Epoch 6/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9568 - loss: 0.0875 - val_accuracy: 0.9021 - val_loss: 0.3164
Epoch 7/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9498 - loss: 0.1160 - val_accuracy: 0.9150 - val_loss: 0.2569
Epoch 8/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9660 - loss: 0.0738 - val_accuracy: 0.9096 - val_lo